In [11]:
import pandas as pd
import kagglehub
import os
import re

In [ ]:
path = kagglehub.dataset_download("nileshmalode1/samsum-dataset-text-summarization")

print("Path to dataset files:", path)

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/mayurimamdi/.cache/kagglehub/datasets/nileshmalode1/samsum-dataset-text-summarization/versions/1


In [5]:
os.listdir(path)

['samsum-train.csv',
 'samsum-validation.csv',
 'samsum_dataset',
 'samsum-test.csv']

In [6]:
df = pd.read_csv(os.path.join(path,'samsum-train.csv'))

In [7]:
df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [8]:
df.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [9]:
df.dropna(inplace=True)

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df['dialogue'] = df['dialogue'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)

In [14]:
df['summary'] = 'startseq ' + df['summary'] + ' endseq'

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer

dialogue_tokenizer = Tokenizer()
dialogue_tokenizer.fit_on_texts(df['dialogue'])

summary_tokenizer = Tokenizer()
summary_tokenizer.fit_on_texts(df['summary'])

In [16]:
dialogue_seq = dialogue_tokenizer.texts_to_sequences(df['dialogue'])
summary_seq = summary_tokenizer.texts_to_sequences(df['summary'])

In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_dialogue_len = max(len(x) for x in dialogue_seq)
max_summary_len = max(len(x) for x in summary_seq)

dialogue_pad = pad_sequences(dialogue_seq, maxlen=max_dialogue_len, padding='post')
summary_pad = pad_sequences(summary_seq, maxlen=max_summary_len, padding='post')

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dialogue_pad, summary_pad, test_size=0.2, random_state=42
)

In [19]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

vocab_dialogue = len(dialogue_tokenizer.word_index) + 1
vocab_summary = len(summary_tokenizer.word_index) + 1

latent_dim = 256

encoder_inputs = Input(shape=(max_dialogue_len,))
enc_emb = Embedding(vocab_dialogue, 100)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

In [24]:
decoder_inputs = Input(shape=(max_summary_len-1,))

dec_emb_layer = Embedding(vocab_summary, 100)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_summary, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [25]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 774)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 61)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 774, 100)  │ 12,525,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 61, 100)   │  1,704,200 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    365,568 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 61, 256), │    365,568 │ embedding_2[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 61, 17042) │  4,379,794 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,340,330 (73.78 MB)

 Trainable params: 19,340,330 (73.78 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
decoder_input = y_train[:, :-1]
decoder_output = y_train[:, 1:]

In [27]:
model.fit(
    [X_train, decoder_input],
    decoder_output,
    batch_size=64,
    epochs=10,
    validation_split=0.2
)

Epoch 1/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 417s 3s/step - loss: 3.1841 - val_loss: 2.2978
Epoch 2/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 384s 3s/step - loss: 2.3025 - val_loss: 2.2257
Epoch 3/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 385s 3s/step - loss: 2.2179 - val_loss: 2.1648
Epoch 4/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - loss: 2.1458 - val_loss: 2.1098
Epoch 5/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 344s 2s/step - loss: 2.0732 - val_loss: 2.0525
Epoch 6/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 355s 2s/step - loss: 2.0022 - val_loss: 2.0071
Epoch 7/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 380s 3s/step - loss: 1.9436 - val_loss: 1.9754
Epoch 8/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 396s 3s/step - loss: 1.8935 - val_loss: 1.9501
Epoch 9/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 367s 2s/step - loss: 1.8493 - val_loss: 1.9332
Epoch 10/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1280s 9s/step - loss: 1.8108 - val_loss: 1.9199


In [41]:
decoder_input_test = y_test[:, :-1]
decoder_output_test = y_test[:, 1:]

In [42]:
model.evaluate(
    [X_test, decoder_input_test],
    decoder_output_test
)

93/93 ━━━━━━━━━━━━━━━━━━━━ 46s 491ms/step - accuracy: 0.7090 - loss: 1.9570


[1.9570436477661133, 0.7089900374412537]

In [43]:
print(decode_sequence(X_test[0].reshape(1, max_dialogue_len)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
 the new team is in the new day in the office
